In [5]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.4 MB/s eta 0:00:00


In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
file = list(uploaded.keys())[0]

df = pd.read_csv(file)
df = df.dropna(subset=["lemma_text"])

print("Docs:", len(df))
df.head()

Saving processed_v3_lemma.csv to processed_v3_lemma.csv
Docs: 7142


,sent_id,processed_text,lemma_text,upos_seq
0,NSDC_UA_28_Feb2014-1,"Шановні колеги , Рада національної безпеки має...","шановний колега , рада національний безпека ма...",ADJ NOUN PUNCT NOUN ADJ NOUN VERB NOUN NOUN AD...
1,NSDC_UA_28_Feb2014-10,"Рішень , які захистять Україну , які зможуть л...","рішення , який захистити Україна , який змогти...",NOUN PUNCT DET VERB PROPN PUNCT DET VERB VERB ...
2,NSDC_UA_28_Feb2014-100,Наших - це в основному військово-морські сили ...,наш — це в основне військово-морський сила — 1...,DET PUNCT PRON ADP NOUN ADJ NOUN PUNCT NUM NOU...
3,NSDC_UA_28_Feb2014-101,А крім морських сил у нас немає ніяких сухопут...,а крім морський сила у ми немає ніякий сухопут...,CCONJ ADP ADJ NOUN ADP PRON VERB DET ADJ PUNCT
4,NSDC_UA_28_Feb2014-102,15 тисяч - це повний склад військових .,15 тисяча — це повний склад військовий .,NUM NOUN PUNCT PRON ADJ NOUN NOUN PUNCT


In [2]:
texts = df["lemma_text"].astype(str)

texts = texts[texts.str.split().str.len() > 5]

print("Docs after filtering:", len(texts))

Docs after filtering: 5338


In [3]:
import re

def tokenize(text):
    return [w for w in re.findall(r"[а-яА-Яіїєґa-zA-Z']+", text.lower())]

sentences = texts.apply(tokenize).tolist()

print(sentences[0][:20])

['шановний', 'колега', 'рада', 'національний', 'безпека', 'мати', 'секретар', 'рада', 'національний', 'безпека', 'найближчий', 'час', 'бути', 'сформований', 'фактично', 'весь', 'підрозділ', 'рнбо', 'призначений', 'заступник']


In [6]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,
    workers=4
)

print("Word2Vec trained")

Word2Vec trained


In [7]:
from gensim.models import FastText

ft_model = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,
    workers=4
)

print("FastText trained")

FastText trained


In [8]:
def show_neighbors(model, word, topn=10):
    if word in model.wv:
        return model.wv.most_similar(word, topn=topn)
    else:
        return "OOV"

In [10]:
words = [
    "закон",
    "україна",
    "рада",
    "проект",
    "постанова",
    "фракція",
    "голосування",
    "рішення",
    "пропозиція",
    "комітет"
]

for word in words:
    print("\n" + "="*50)
    print(f"WORD: {word}")

    print("\nWord2Vec:")
    print(show_neighbors(w2v_model, word))

    print("\nFastText:")
    print(show_neighbors(ft_model, word))


WORD: закон

Word2Vec:
[('зміна', 0.9630680680274963), ('внесення', 0.9467073082923889), ('деякий', 0.9422266483306885), ('конституція', 0.940680980682373), ('кодекс', 0.9243055582046509), ('чинність', 0.9152517914772034), ('розділ', 0.8980656862258911), ('денний', 0.8942438960075378), ('указ', 0.894159197807312), ('редакція', 0.8877537250518799)]

FastText:
[('зміна', 0.9840596318244934), ('змі', 0.9695832133293152), ('законодавчо', 0.9656999111175537), ('про', 0.9593562483787537), ('конституція', 0.9539563059806824), ('порядок', 0.9532231092453003), ('згідно', 0.9469157457351685), ('внесок', 0.9453120827674866), ('внесення', 0.9447486400604248), ('проект', 0.9445458054542542)]

WORD: україна

Word2Vec:
[('сдпу', 0.8633309006690979), ('четвертий', 0.8607116341590881), ('невідкладно', 0.8533045649528503), ('сесія', 0.8524160385131836), ('о', 0.8524119257926941), ('звернення', 0.8465362787246704), ('апарат', 0.8426750302314758), ('подання', 0.842205822467804), ('подати', 0.838464915752